# Data Skew Optimization: Salting, Skew Hints, and AQE Skew Joins

**Module**: 10_performance_and_optimization  
**Notebook**: 06_data_skew_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Comprehensive hands-on laboratory for diagnosing and fixing data skew: key salting, broadcast join avoidance, and AQE automatic skew splitting.
- **Why does this topic exist?**: Data skew is the primary cause of jobs hanging at 99% progress.
- **What problem does it solve?**: 1 task running for 40 minutes while all other tasks finished in seconds.
- **Where is it used in real-world Data Engineering?**: Handling null keys and power-law customer purchase distributions.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Data skew basics, joins, AQE |
| **Required depth** | Working knowledge: can choose and verify a skew remediation in production |
| **Data Engineer** | Apply the right skew technique for joins and aggregations. |
| **Senior Data Engineer** | Decide between broadcast, salting, AQE, and hot-key isolation. |
| **Spark specialist** | Reason about skew hints, partition splitting, and correctness after key modification. |
| **Next topic** | `10_performance_and_optimization/07_performance_benchmarking.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Skew remediation is a decision tree, not a single trick. If one side is small, broadcast it. If the hot keys
are known and few, isolate them. If the join is eligible, AQE can split skewed partitions. Salting is a general
fallback but adds a second aggregation and changes the key, so correctness must be verified. More partitions alone
rarely fix a single hot key.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Skew hint** | Tell Spark about a hot key | `/*+ SKEW('table', 'column') */` |
| **AQE skew join** | Runtime partition splitting | Automatic for eligible joins |
| **Salting** | Spread a hot key | Add a random suffix and aggregate again |
| **Isolation** | Process hot keys separately | Split, compute, and union |
| **Broadcast** | Remove the shuffle | Best when one side fits in memory |
| **Partition count** | Parallelism | Does not split a single hot key by itself |

### How it works, step by step

1. Detect skew from task duration and shuffle read.
2. Identify the hot keys and their business meaning.
3. Check whether one join side is small enough to broadcast.
4. Check whether AQE skew handling is eligible.
5. If not, isolate the hot keys or salt the key.
6. Verify row counts and aggregation correctness after the change.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Skew Remediation Playbook: 1. Broadcast Join -> 2. Salting -> 3. AQE Skew Join -> 4. Isolate Skewed Keys

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Skew_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Create skewed dataset
skewed = spark.range(0, 100000).withColumn("key", F.when(F.col("id") < 90000, "HOT_KEY").otherwise(F.concat(F.lit("KEY_"), F.col("id"))))
print("Key distribution:")
skewed.groupBy("key").count().orderBy(F.desc("count")).show(3)

---

## Experiment: Key Salting Implementation

- **1. What are we testing?**: Salt the skewed key to distribute the hotspot across 8 independent partitions.
- **2. What do we expect?**: The single massive partition is split into 8 uniform slices.

In [ ]:
salted = skewed.withColumn("salted_key", F.concat(F.col("key"), F.lit("_"), F.floor(F.rand() * 8)))
print("Salted key breakdown:")
salted.filter("key = 'HOT_KEY'").groupBy("salted_key").count().show()

**3. What happened?**: The 90,000 HOT_KEY rows were distributed across 8 salted keys (~11,250 rows each).  
**4. Why did it happen?**: Random salt assigned rows across distinct hash partitions.  
**5. Practical Engineering Lesson**: Salting converts an asymmetric bottleneck into uniform parallel task waves.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Viral content aggregation

- **Industry**: Digital media
- **Scenario**: A daily aggregation is dominated by one content ID that receives most events.
- **Problem**: One reducer processes the hot key while the rest finish early.
- **Why this concept is relevant**: Salting the content key spreads the hot key across reducers, followed by a second aggregation.
- **Implementation idea**: Measure max task duration before and after; verify the final counts match.
- **Expected result**: A balanced stage with the same business result.
- **Alternative**: Isolate the hot content and union the result.
- **Trade-offs**: Isolation is simpler for a known key but requires maintaining a special-case path.

### Use Case 2: Skewed merchant join

- **Industry**: Payments
- **Scenario**: A fraud feature join is slow because one merchant has millions of transactions.
- **Problem**: The merchant dimension is small enough to broadcast, but the join still shuffles.
- **Why this concept is relevant**: A broadcast join removes the shuffle and avoids the hot-key partition entirely.
- **Implementation idea**: Check the dimension size, enable broadcast, and monitor executor memory.
- **Expected result**: A join with no large-side shuffle.
- **Alternative**: Salt the merchant key.
- **Trade-offs**: Salting adds a second aggregation and is unnecessary when broadcast is feasible.

---

## Business Value

- Predictable SLAs for skew-prone pipelines.
- Lower compute waste from idle executors.
- Correct results after key transformations.
- A repeatable remediation playbook.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Try broadcast first when one side is small.
- Use AQE skew handling when the join is eligible.
- Isolate known hot keys for predictable performance.
- Salt only when simpler options fail, and verify correctness.
- Measure max/median task duration, not only average.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare skew remedies

**Steps**

1. Create a dataset with one hot key and run a join.
2. Record max/median task duration.
3. Apply broadcast, then salting, and compare each result.

**What to observe**: Broadcast removes the shuffle; salting spreads the key but adds a second aggregation.

**Healthy result**: The chosen remedy balances the stage and preserves row counts.

**Problematic result**: The stage remains skewed because the salt was not applied to both sides.

**Engineering takeaway**: Choose the simplest remedy that preserves correctness.

---

## Trade-offs and Alternatives

### Advantages

- Large latency improvements
- Multiple techniques
- AQE automation

### Limitations

- Salting adds a shuffle
- AQE eligibility is limited
- Hints can become stale

### When to use it

- Hot-key joins
- Skewed aggregations
- Null-heavy joins

### When not to use it

- Adding partitions only
- Salting every key
- Dropping hot keys without a business rule

### Alternatives inside Spark

- AQE skew join
- SKEW hint
- Salting
- Broadcast

### Alternatives outside Spark

- Pre-aggregation
- Custom partitioners
- Streaming key parallelism

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Skew Column
**Problem Statement**: Profile top keys in dirty dataset.  
**Input / Context**: dirty_df

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Filter Skewed Nulls
**Problem Statement**: Isolate null keys prior to join.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): AQE Skew Join Trigger
**Problem Statement**: Threshold settings for AQE skew join.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The salt that doubled the rows

**Scenario**: After salting a join key, the output contains duplicate rows.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether both sides were salted consistently.
2. Verify the join condition includes the salt column.
3. Confirm the post-join aggregation restores the original grain.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Skew Column):
```python
spark.read.option('header', True).csv('/opt/spark-data/quality/orders_dirty.csv').groupBy('customer_id').count().orderBy(F.desc('count')).show(5)
```

#### Solution 2 (Filter Skewed Nulls):
```python
print('Filter isNotNull() on left and right keys; union null records separately.')
```

#### Solution 3 (AQE Skew Join Trigger):
```python
print('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes (default 256MB)')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Salting Formula**: key +'_'+ rand(0, K).
- **AQE Fix**: AQE automatically splits skewed shuffle partitions into sub-partitions.

### Essential Technical Terms
- `Data Skew`: Asymmetric record distribution.
- `Salting`: Adding random prefix to split key hotspots.

### Common Engineering Mistakes
- **Assuming more memory fixes skew**: If all rows map to 1 key, 1 task will still process them sequentially.

### Interview Questions on This Topic
- **Q: Explain how salting resolves a data skew join bottleneck.**
  - *A*: Salting appends a random number to the skewed key of the large table and replicates the matching small table keys with matching numbers, distributing the hotspot across multiple parallel partitions.


### More Interview Questions

- **Q: Why does adding partitions not fix a single hot key?**
  - *A*: All records for the same key hash to the same partition, so increasing the partition count does not split that key's data.
- **Q: When is a SKEW hint appropriate?**
  - *A*: When you know a specific key is hot and the join is eligible; the hint helps the optimizer plan for the skew, but it is not a substitute for measuring the result.

### What to Study Next
Open 07_performance_benchmarking.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')